# Exercise 4: Transformers on Images + GLU-MLP Ablations (ViT × GLU Variants)

## In this exercise you will combine two influential ideas:

Vision Transformers (ViT) from “An Image is Worth 16×16 Words: Transformers for Image Recognition at Scale” (Dosovitskiy et al., 2020) https://arxiv.org/pdf/2010.11929:
ViT shows that you can treat an image like a sequence of tokens by splitting it into non-overlapping patches (e.g. 16×16 in the paper), embedding each patch into a vector, adding positional information, and then applying standard Transformer blocks for classification.

Gated MLPs (GLU variants) from “GLU Variants Improve Transformer” (Shazeer, 2020) https://arxiv.org/pdf/2002.05202:
Shazeer proposes replacing the standard Transformer feed-forward layer (FFN/MLP) with gated linear unit (GLU) variants such as GEGLU and SwiGLU, which often improves training dynamics and final performance under comparable compute/parameter budgets.

## What you will do

You will implement a tiny ViT-style classifier for MNIST, then run a controlled ablation where you replace the MLP inside each Transformer block:

Baseline FFN (GELU):
Linear(d_model → d_ff) → GELU → Linear(d_ff → d_model)

GLU-family MLPs (choose at least two and justify):

GEGLU, SwiGLU, other activation functions

Your goal is to evaluate whether these GLU variants change:

- convergence speed (loss vs steps),

- final test accuracy,

- and/or stability across runs.

## Key ViT concepts you will implement

- To convert MNIST images into Transformer tokens, you will:
  Patchify each 28×28 image into non-overlapping P×P patches.
  If P=4, then you get a 7×7 patch grid → 49 tokens per image.

- Embed patches with a linear layer: patch vectors → d_model.

- Add positional embeddings so the model knows where each patch came from.

- Apply n_layers Transformer encoder blocks.

- Pool token features (e.g., mean pooling) and project to 10 classes.

## Key GLU concept you will implement

GLU-style MLPs replace a standard FFN with a gating mechanism:
compute two projections a and b, apply a nonlinearity to a (variant-dependent), multiply elementwise: act(a) * b, project back to d_model.
To keep the comparison fair, use the 2/3 width rule from Shazeer.

What we provide vs what you implement

### We provide:

- MNIST loading + dataloaders

- a minimal training loop structure (AdamW)

- a suggested small model configuration that runs on CPU

### You implement:

- patch tokenization (patchify)

- patch embedding + positional embedding strategy

- a pre-LN Transformer encoder block using nn.MultiheadAttention

- at least two GLU MLP variants + one FFN baseline

- metric logging sufficient to support your conclusion

## Deliverables

Run at least 3 variants (baseline + the activation functions you choose for GLU) and report:

- final and best test accuracy

- number of trainable parameters

- a plot or printed summary of loss/accuracy over epochs

- a short discussion of your results

In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass, replace
from time import perf_counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
def patchify(x: torch.Tensor, patch_size: int) -> torch.Tensor:
    """Convert images to patch tokens."""
    b, c, h, w = x.shape
    assert h % patch_size == 0 and w % patch_size == 0, "Image size must be divisible by patch size"

    gh = h // patch_size
    gw = w // patch_size

    patches = x.unfold(2, patch_size, patch_size).unfold(3, patch_size, patch_size)
    patches = patches.permute(0, 2, 3, 1, 4, 5).contiguous()
    patches = patches.view(b, gh * gw, c * patch_size * patch_size)
    return patches

In [3]:
# TODO: Add positional encoding as done in the ViT paper and patch projection
class PatchEmbed(nn.Module):
    def __init__(self, patch_dim: int, d_model: int):
        super().__init__()
        self.proj = nn.Linear(patch_dim, d_model)

    def forward(self, x_patches: torch.Tensor) -> torch.Tensor:
        return self.proj(x_patches)


class PositionalEmbedding(nn.Module):
    def __init__(self, num_tokens: int, d_model: int):
        super().__init__()
        self.pos = nn.Parameter(torch.zeros(1, num_tokens, d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pos

In [4]:
# TODO: Define the variants you want to compare against each other from the GLU paper. Justify your choice.
class FeedForward(nn.Module):
    """
    Standard Transformer FFN:
      x -> Linear(d_model->d_ff) -> GELU -> Dropout -> Linear(d_ff->d_model) -> Dropout
    """
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x


class GLUFeedForward(nn.Module):
    """GLU-family FFN"""
    def __init__(self, d_model: int, d_ff_gated: int, dropout: float, variant: str):
        super().__init__()
        self.variant = variant.lower()
        self.w_act = nn.Linear(d_model, d_ff_gated)
        self.w_gate = nn.Linear(d_model, d_ff_gated)
        self.w_out = nn.Linear(d_ff_gated, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        a = self.w_act(x)
        b = self.w_gate(x)

        if self.variant == "geglu":
            h = F.gelu(a)
        elif self.variant == "swiglu":
            h = F.silu(a)
        elif self.variant == "reglu":
            h = F.relu(a)
        else:
            raise ValueError(f"Unknown GLU variant: {self.variant}")

        x = h * b
        x = self.dropout(x)
        x = self.w_out(x)
        x = self.dropout(x)
        return x

In [5]:
class TransformerEncoderBlock(nn.Module):
    """
    Pre-LN encoder block:
      x = x + Dropout(SelfAttn(LN(x)))
      x = x + Dropout(MLP(LN(x)))
    """
    def __init__(self, d_model: int, n_heads: int, mlp: nn.Module, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.mlp = mlp
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        attn_in = self.ln1(x)
        attn_out, _ = self.attn(attn_in, attn_in, attn_in, need_weights=False)
        x = x + self.dropout(attn_out)

        mlp_in = self.ln2(x)
        x = x + self.dropout(self.mlp(mlp_in))
        return x

In [6]:
class TinyViT(nn.Module):
    """
    Tiny ViT-style classifier for MNIST.
    - patchify -> patch embed -> pos embed -> blocks -> mean pool -> head
    """
    def __init__(
        self,
        patch_size: int,
        d_model: int,
        n_heads: int,
        n_layers: int,
        d_ff: int,
        dropout: float,
        mlp_kind: str,
    ):
        super().__init__()
        assert 28 % patch_size == 0
        grid = 28 // patch_size
        self.num_tokens = grid * grid
        self.patch_size = patch_size
        patch_dim = patch_size * patch_size

        self.patch_embed = PatchEmbed(patch_dim=patch_dim, d_model=d_model)
        self.pos_embed = PositionalEmbedding(num_tokens=self.num_tokens, d_model=d_model)

        def make_mlp() -> nn.Module:
            if mlp_kind == "ffn_gelu":
                return FeedForward(d_model=d_model, d_ff=d_ff, dropout=dropout)
            if mlp_kind in {"geglu", "swiglu", "reglu"}:
                d_ff_gated = int((2 * d_ff) / 3)
                return GLUFeedForward(d_model=d_model, d_ff_gated=d_ff_gated, dropout=dropout, variant=mlp_kind)
            raise ValueError(f"Unknown mlp_kind: {mlp_kind}")

        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                mlp=make_mlp(),
                dropout=dropout,
            )
            for _ in range(n_layers)
        ])

        self.head = nn.Linear(d_model, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = patchify(x, self.patch_size)
        x = self.patch_embed(x)
        x = self.pos_embed(x)

        for block in self.blocks:
            x = block(x)

        x = x.mean(dim=1)
        logits = self.head(x)
        return logits

In [7]:
@dataclass(frozen=True)
class TrainConfig:
    seed: int = 0
    batch_size: int = 128
    epochs: int = 3
    lr: float = 3e-4
    weight_decay: float = 0.01
    device: str = "cpu"  # set "cuda" if available

In [ ]:
def train_one_run(
    mlp_kind: str,
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    cfg: TrainConfig,
) -> dict:
    torch.manual_seed(cfg.seed)
    model.to(cfg.device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    train_losses: list[float] = []
    test_accs: list[float] = []
    epoch_times_sec: list[float] = []

    run_start = perf_counter()
    for epoch in range(cfg.epochs):
        epoch_start = perf_counter()

        # Train loop
        model.train()
        for i, (xb, yb) in enumerate(train_loader):
            xb = xb.to(cfg.device)
            yb = yb.to(cfg.device)

            logits = model(xb)
            loss = F.cross_entropy(logits, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

            train_losses.append(loss.item())

        # Evaluation loop NOTE: Should be no need to change this
        model.eval()
        correct = 0.0
        total = 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(cfg.device)
                yb = yb.to(cfg.device)
                logits = model(xb)
                correct += (logits.argmax(dim=-1) == yb).float().sum().item()
                total += yb.numel()

        test_accs.append(correct / total)
        epoch_times_sec.append(perf_counter() - epoch_start)
        print(f"[{mlp_kind}] seed={cfg.seed} epoch {epoch+1}/{cfg.epochs} | test acc: {test_accs[-1]:.4f}")

    runtime_sec = perf_counter() - run_start
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        "mlp_kind": mlp_kind,
        "seed": cfg.seed,
        "n_params": n_params,
        "train_losses": train_losses,
        "test_accs": test_accs,
        "final_test_acc": test_accs[-1],
        "best_test_acc": max(test_accs),
        "runtime_sec": runtime_sec,
        "epoch_times_sec": epoch_times_sec,
    }

### Variant choice (GLU paper)

For the ablation we compare:
- `ffn_gelu` (standard Transformer FFN baseline)
- `geglu` (GLU variant with GELU gate activation)
- `swiglu` (GLU variant with SiLU gate activation)

Why these variants:
- They are GLU choices discussed in the GLU paper, they are used in the transformers implementations
- `ffn_gelu` provides a non-gated baseline under the same training setup.
- Using `geglu` and `swiglu` tests whether gating improves convergence and final accuracy at similar model scale (using the 2/3 width rule for gated FFNs).

In [ ]:
cfg = TrainConfig(seed=0, batch_size=128, epochs=3, lr=3e-4, weight_decay=0.01, device="cpu")

tfm = transforms.Compose([transforms.ToTensor()])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

# Tiny model example. TODO: You're welcome to experiment with these parameters
patch_size = 4
d_model = 64
n_heads = 4
n_layers = 2
d_ff = 256
dropout = 0.1

# Variants selected from GLU paper family: GELU baseline + GEGLU + SwiGLU
runs = ["ffn_gelu", "geglu", "swiglu"]
seeds = [0, 1, 2]
results = []

for seed in seeds:
    run_cfg = replace(cfg, seed=seed)
    print(f"\n=== Seed {seed} ===")
    for kind in runs:
        model = TinyViT(
            patch_size=patch_size,
            d_model=d_model,
            n_heads=n_heads,
            n_layers=n_layers,
            d_ff=d_ff,
            dropout=dropout,
            mlp_kind=kind,
        )
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Run: {kind} | params: {n_params}")
        out = train_one_run(kind, model, train_loader, test_loader, run_cfg)
        results.append(out)

print("\nAggregate summary (mean ± std over seeds):")
for kind in runs:
    rows = [r for r in results if r["mlp_kind"] == kind]
    finals = torch.tensor([r["final_test_acc"] for r in rows], dtype=torch.float32)
    bests = torch.tensor([r["best_test_acc"] for r in rows], dtype=torch.float32)
    times = torch.tensor([r["runtime_sec"] for r in rows], dtype=torch.float32)
    params = rows[0]["n_params"]
    print(
        f"{kind:<8} | params={params:<8d} | "
        f"final={finals.mean():.4f}±{finals.std(unbiased=False):.4f} | "
        f"best={bests.mean():.4f}±{bests.std(unbiased=False):.4f} | "
        f"runtime={times.mean():.1f}s±{times.std(unbiased=False):.1f}s"
    )

In [10]:
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

# expects `results` from the training cell
if len(results) == 0:
    print("No results found. Run the experiment cell first.")
else:
    grouped: dict[str, list[dict]] = defaultdict(list)
    for r in results:
        grouped[r["mlp_kind"]].append(r)

    preferred_order = ["ffn_gelu", "geglu", "swiglu"]
    kinds = [k for k in preferred_order if k in grouped]

    def moving_avg(values: list[float], window: int = 100) -> list[float]:
        if len(values) == 0:
            return []
        out = []
        run_sum = 0.0
        for i, v in enumerate(values):
            run_sum += v
            if i >= window:
                run_sum -= values[i - window]
            out.append(run_sum / min(i + 1, window))
        return out

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))

    # (1) Smoothed train loss per step, mean ± std over seeds
    for kind in kinds:
        curves = [moving_avg(r["train_losses"], window=100) for r in grouped[kind]]
        min_len = min(len(c) for c in curves)
        arr = np.array([c[:min_len] for c in curves], dtype=float)
        mean = arr.mean(axis=0)
        std = arr.std(axis=0)
        x = np.arange(min_len)
        axes[0, 0].plot(x, mean, label=kind)
        axes[0, 0].fill_between(x, mean - std, mean + std, alpha=0.2)
    axes[0, 0].set_title("Train loss (smoothed, mean±std)")
    axes[0, 0].set_xlabel("step")
    axes[0, 0].set_ylabel("loss")
    axes[0, 0].legend(fontsize=9)

    # (2) Test accuracy per epoch, mean ± std over seeds
    for kind in kinds:
        arr = np.array([r["test_accs"] for r in grouped[kind]], dtype=float)
        mean = arr.mean(axis=0)
        std = arr.std(axis=0)
        epochs = np.arange(1, arr.shape[1] + 1)
        axes[0, 1].plot(epochs, mean, marker="o", label=kind)
        axes[0, 1].fill_between(epochs, mean - std, mean + std, alpha=0.2)
    axes[0, 1].set_title("Test accuracy per epoch (mean±std)")
    axes[0, 1].set_xlabel("epoch")
    axes[0, 1].set_ylabel("accuracy")
    axes[0, 1].set_ylim(0.0, 1.0)
    axes[0, 1].legend(fontsize=9)

    # (3) Best accuracy distribution across seeds
    best_data = [[r["best_test_acc"] for r in grouped[k]] for k in kinds]
    axes[0, 2].boxplot(best_data, labels=kinds, showmeans=True)
    axes[0, 2].set_title("Best test accuracy across seeds")
    axes[0, 2].set_ylabel("best accuracy")
    axes[0, 2].set_ylim(0.0, 1.0)

    # (4) Runtime per run (mean ± std)
    runtime_means = []
    runtime_stds = []
    for kind in kinds:
        vals = np.array([r["runtime_sec"] for r in grouped[kind]], dtype=float)
        runtime_means.append(vals.mean())
        runtime_stds.append(vals.std())
    axes[1, 0].bar(kinds, runtime_means, yerr=runtime_stds, capsize=4)
    axes[1, 0].set_title("Runtime per run (seconds)")
    axes[1, 0].set_ylabel("seconds")

    # (5) Params vs best accuracy (all runs)
    for kind in kinds:
        xs = [r["n_params"] for r in grouped[kind]]
        ys = [r["best_test_acc"] for r in grouped[kind]]
        axes[1, 1].scatter(xs, ys, alpha=0.8, label=kind)
    axes[1, 1].set_title("Params vs best accuracy")
    axes[1, 1].set_xlabel("# trainable params")
    axes[1, 1].set_ylabel("best accuracy")
    axes[1, 1].set_ylim(0.0, 1.0)
    axes[1, 1].ticklabel_format(style="sci", axis="x", scilimits=(0, 0))
    axes[1, 1].legend(fontsize=9)

    # (6) Delta accuracy vs GELU baseline
    base = np.array([r["test_accs"] for r in grouped["ffn_gelu"]], dtype=float).mean(axis=0)
    epochs = np.arange(1, len(base) + 1)
    for kind in kinds:
        arr = np.array([r["test_accs"] for r in grouped[kind]], dtype=float)
        delta = arr.mean(axis=0) - base
        axes[1, 2].plot(epochs, delta, marker="o", label=kind)
    axes[1, 2].axhline(0.0, color="black", linewidth=1)
    axes[1, 2].set_title("Delta accuracy vs ffn_gelu")
    axes[1, 2].set_xlabel("epoch")
    axes[1, 2].set_ylabel("accuracy delta")
    axes[1, 2].legend(fontsize=9)

    plt.tight_layout()
    plt.show()

    print("\nRun summary (mean ± std over seeds):")
    for kind in kinds:
        rows = grouped[kind]
        finals = np.array([r["final_test_acc"] for r in rows], dtype=float)
        bests = np.array([r["best_test_acc"] for r in rows], dtype=float)
        times = np.array([r["runtime_sec"] for r in rows], dtype=float)
        params = rows[0]["n_params"]
        print(
            f"{kind:<8} | params={params:<8d} | "
            f"final={finals.mean():.4f}±{finals.std():.4f} | "
            f"best={bests.mean():.4f}±{bests.std():.4f} | "
            f"runtime={times.mean():.1f}s±{times.std():.1f}s"
        )

No results found. Run the experiment cell first.
